# Deforestation parameterisation

Run the last cell. It opens a local page with sliders; each change redraws the map from the Sentinel-1 and forest-baseline GeoTIFFs already on disk. The slider starting values are the constants in the setup cell. This does not call openEO and does not rerun Sentinel-1.

A pixel is deforested when all of these hold:

- `sd >= temporal_variability_threshold`
- `(p95 - p05) / abs(p05) >= flattening_threshold`
- `min_sse <= logistic_sse_threshold`
- forest baseline `B0 == 1`

`min_connected_area` then drops 8-connected remaining-forest patches smaller than that area. The nearest-neighbour year fill from the batch job is not applied.

This is a local preview. It will not match a batch job pixel for pixel: openEO resamples inside the process graph and runs the connectivity UDF in overlapping tiles.

In [12]:
import math
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from rasterio.enums import Resampling
from rasterio.warp import reproject

UDF_DIR = Path("../udf").resolve()
if str(UDF_DIR) not in sys.path:
    sys.path.insert(0, str(UDF_DIR))
from connectivity_mask import small_region_mask

S1_TIF = Path("../s1-processing/output-udp/openEO.tif")
FOREST_BASELINE_TIF = Path("../forest-baseline/output-udp/openEO.tif")

SPATIAL_RESOLUTION_M = 30.0
PIXEL_AREA_HA = (SPATIAL_RESOLUTION_M ** 2) / 10_000.0

TEMPORAL_VARIABILITY_THRESHOLD = 0.6  # dB, applied to sd
FLATTENING_THRESHOLD = 0.14  # dimensionless
LOGISTIC_SSE_THRESHOLD = 18.3  # dB^2, applied to min_sse
MIN_CONNECTED_AREA_M2 = 10_000.0

In [13]:
def read_band(dataset: rasterio.DatasetReader, name: str) -> np.ndarray:
    descriptions = list(dataset.descriptions)
    if name not in descriptions:
        raise ValueError(f"{dataset.name} has bands {descriptions}, missing {name!r}")
    return dataset.read(descriptions.index(name) + 1).astype(np.float32)


def load_inputs(s1_path: Path, forest_path: Path) -> dict[str, np.ndarray]:
    if not s1_path.is_file():
        raise FileNotFoundError(s1_path)
    if not forest_path.is_file():
        raise FileNotFoundError(forest_path)

    with rasterio.open(s1_path) as s1, rasterio.open(forest_path) as forest:
        if s1.crs != forest.crs:
            raise ValueError(f"CRS mismatch: S1 {s1.crs}, forest baseline {forest.crs}")

        bands = {
            name: read_band(s1, name)
            for name in ("sd", "p05", "p95", "min_sse", "min_sse_t")
        }
        forest_src = forest.read(1)
        forest_nodata = 255 if forest.nodata is None else int(forest.nodata)
        forest_on_s1 = np.full(bands["sd"].shape, forest_nodata, dtype=np.uint8)
        reproject(
            source=forest_src,
            destination=forest_on_s1,
            src_transform=forest.transform,
            src_crs=forest.crs,
            src_nodata=forest_nodata,
            dst_transform=s1.transform,
            dst_crs=s1.crs,
            dst_nodata=forest_nodata,
            resampling=Resampling.nearest,
        )

    valid = (
        np.isfinite(bands["sd"])
        & np.isfinite(bands["p05"])
        & np.isfinite(bands["p95"])
        & np.isfinite(bands["min_sse"])
        & (forest_on_s1 != forest_nodata)
    )
    bands["forest"] = valid & (forest_on_s1 == 1)
    bands["valid"] = valid
    return bands


INPUTS = load_inputs(S1_TIF, FOREST_BASELINE_TIF)
print(
    f"grid {INPUTS['sd'].shape[1]} x {INPUTS['sd'].shape[0]} px, "
    f"valid {int(INPUTS['valid'].sum())}, forest baseline {int(INPUTS['forest'].sum())}"
)

grid 2526 x 1773 px, valid 2821132, forest baseline 707921


In [14]:
import io
import threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import parse_qs, urlparse

import scipy.ndimage as ndi
from matplotlib.patches import Rectangle

from IPython.display import HTML, display

DISPLAY_MAX_PX = 800

CLASS_NODATA = 0
CLASS_NON_FOREST = 1
CLASS_FOREST = 2
CLASS_DEFORESTED = 3

CLASS_CMAP = ListedColormap(["#00000000", "#c4a574", "#1b7a3a", "#d62828"])
CLASS_LEGEND = [
    Patch(facecolor="#c4a574", edgecolor="none", label="non-forest"),
    Patch(facecolor="#1b7a3a", edgecolor="none", label="remaining forest"),
    Patch(facecolor="#d62828", edgecolor="none", label="deforested"),
]


def classify(
    temporal_variability_threshold: float,
    flattening_threshold: float,
    logistic_sse_threshold: float,
    min_connected_area_m2: float,
) -> tuple[np.ndarray, int]:
    sd = INPUTS["sd"]
    p05 = INPUTS["p05"]
    p95 = INPUTS["p95"]
    min_sse = INPUTS["min_sse"]
    valid = INPUTS["valid"]
    forest = INPUTS["forest"]

    flattening = (p95 - p05) / np.abs(p05)
    event = (
        forest
        & (sd >= temporal_variability_threshold)
        & np.isfinite(flattening)
        & (flattening >= flattening_threshold)
        & (min_sse <= logistic_sse_threshold)
    )
    remaining = forest & ~event

    if min_connected_area_m2 > 0:
        min_pixels = math.ceil(min_connected_area_m2 / (SPATIAL_RESOLUTION_M ** 2))
        if min_pixels >= 1:
            drop = small_region_mask(np.ascontiguousarray(remaining), int(min_pixels))
            remaining = remaining & ~drop

    classes = np.full(sd.shape, CLASS_NODATA, dtype=np.uint8)
    classes[valid] = CLASS_NON_FOREST
    classes[remaining] = CLASS_FOREST
    classes[event] = CLASS_DEFORESTED
    return classes, int(event.sum())


def largest_event_window(classes: np.ndarray, pad: int = 50, max_side: int = 420) -> tuple[int, int, int, int] | None:
    event = classes == CLASS_DEFORESTED
    if not np.any(event):
        return None
    labeled, n = ndi.label(event, structure=np.ones((3, 3), dtype=np.int8))
    if n < 1:
        return None
    counts = np.bincount(labeled.ravel())
    counts[0] = 0
    ys, xs = np.nonzero(labeled == int(counts.argmax()))
    y_mid = int(round(float(ys.mean())))
    x_mid = int(round(float(xs.mean())))
    y0 = int(ys.min()) - pad
    y1 = int(ys.max()) + 1 + pad
    x0 = int(xs.min()) - pad
    x1 = int(xs.max()) + 1 + pad
    if (y1 - y0) > max_side:
        y0 = y_mid - max_side // 2
        y1 = y0 + max_side
    if (x1 - x0) > max_side:
        x0 = x_mid - max_side // 2
        x1 = x0 + max_side
    height, width = classes.shape
    y0 = max(0, y0)
    x0 = max(0, x0)
    y1 = min(height, max(y0 + 1, y1))
    x1 = min(width, max(x0 + 1, x1))
    return y0, y1, x0, x1


EDGE_WINDOW = (1244, 1496, 916, 1224)  # y0, y1, x0, x1: forest edge below the main patch


def clip_window(window: tuple[int, int, int, int], shape: tuple[int, int]) -> tuple[int, int, int, int]:
    y0, y1, x0, x1 = window
    height, width = shape
    y0 = max(0, min(height - 1, y0))
    x0 = max(0, min(width - 1, x0))
    y1 = max(y0 + 1, min(height, y1))
    x1 = max(x0 + 1, min(width, x1))
    return y0, y1, x0, x1


def mark_window(ax, window: tuple[int, int, int, int], step: int, color: str) -> None:
    y0, y1, x0, x1 = window
    ax.add_patch(Rectangle(
        (x0 / step - 0.5, y0 / step - 0.5),
        (x1 - x0) / step,
        (y1 - y0) / step,
        fill=False,
        edgecolor=color,
        linewidth=1.2,
    ))


def show_zoom(ax, classes: np.ndarray, window: tuple[int, int, int, int], title: str) -> None:
    y0, y1, x0, x1 = window
    zoom = np.ma.masked_equal(classes[y0:y1, x0:x1], CLASS_NODATA)
    ax.imshow(zoom, cmap=CLASS_CMAP, vmin=0, vmax=3, interpolation="nearest")
    ax.set_axis_off()
    ax.set_title(f"{title}  {x1 - x0}×{y1 - y0} px")


def render_png(
    temporal_variability_threshold: float,
    flattening_threshold: float,
    logistic_sse_threshold: float,
    min_connected_area_m2: float,
) -> bytes:
    classes, n_deforested = classify(
        temporal_variability_threshold,
        flattening_threshold,
        logistic_sse_threshold,
        min_connected_area_m2,
    )
    area_ha = n_deforested * PIXEL_AREA_HA
    step = max(1, math.ceil(max(classes.shape) / DISPLAY_MAX_PX))
    shown = np.ma.masked_equal(classes[::step, ::step], CLASS_NODATA)
    largest = largest_event_window(classes)
    edge = clip_window(EDGE_WINDOW, classes.shape)

    fig, axes = plt.subplots(1, 3, figsize=(16.2, 5.2), dpi=100, layout="constrained")
    axes[0].imshow(shown, cmap=CLASS_CMAP, vmin=0, vmax=3, interpolation="nearest")
    axes[0].set_axis_off()
    axes[0].set_title(
        f"deforested {n_deforested:,} px  ({area_ha:,.1f} ha)\n"
        f"sd ≥ {temporal_variability_threshold:.2f} dB, "
        f"flattening ≥ {flattening_threshold:.2f}, "
        f"min_sse ≤ {logistic_sse_threshold:.1f} dB², "
        f"min area {min_connected_area_m2:.0f} m²"
    )
    axes[0].legend(handles=CLASS_LEGEND, loc="lower right", frameon=True, fontsize=8)
    mark_window(axes[0], edge, step, "#4cc9f0")

    if largest is None:
        axes[1].set_facecolor("#1e1e1e")
        axes[1].set_axis_off()
        axes[1].text(0.5, 0.5, "no deforested pixels", ha="center", va="center", color="white", transform=axes[1].transAxes)
    else:
        mark_window(axes[0], largest, step, "white")
        show_zoom(axes[1], classes, largest, "largest patch")
    show_zoom(axes[2], classes, edge, "forest edge")

    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    plt.close(fig)
    return buf.getvalue()



_PAGE = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>deforestation parameters</title>
<style>
body { font: 14px/1.4 sans-serif; margin: 12px 16px; color: #fff; background: #1e1e1e; }
label { display: grid; grid-template-columns: 140px 280px 72px; gap: 8px; align-items: center; margin: 6px 0; }
output { font-variant-numeric: tabular-nums; color: #fff; background: #111; padding: 1px 6px; border-radius: 3px; }
img { max-width: 100%; margin-top: 8px; image-rendering: pixelated; }
#status { min-height: 1.2em; color: #555; }
</style>
</head>
<body>
<label>sd ≥ dB <input id="sd" type="range" min="0" max="2" step="0.05" value="__SD__"><output id="sd_v"></output></label>
<label>flattening ≥ <input id="flat" type="range" min="0" max="1" step="0.01" value="__FLAT__"><output id="flat_v"></output></label>
<label>min_sse ≤ <input id="sse" type="range" min="0" max="80" step="0.1" value="__SSE__"><output id="sse_v"></output></label>
<label>min area m² <input id="area" type="range" min="0" max="50000" step="1000" value="__AREA__"><output id="area_v"></output></label>
<div id="status"></div>
<img id="map" alt="classification">
<script>
const fields = [
  ["sd", "sd_v", "temporal_variability_threshold"],
  ["flat", "flat_v", "flattening_threshold"],
  ["sse", "sse_v", "logistic_sse_threshold"],
  ["area", "area_v", "min_connected_area_m2"],
];
let seq = 0;
let timer = 0;
function refresh() {
  const my = ++seq;
  const params = new URLSearchParams();
  for (const [id, out, name] of fields) {
    const el = document.getElementById(id);
    document.getElementById(out).value = el.value;
    params.set(name, el.value);
  }
  document.getElementById("status").textContent = "drawing\u2026";
  const next = new Image();
  next.onload = () => {
    if (my !== seq) return;
    document.getElementById("map").src = next.src;
    document.getElementById("status").textContent = "";
  };
  next.onerror = () => {
    if (my === seq) document.getElementById("status").textContent = "draw failed";
  };
  next.src = "/map.png?" + params.toString() + "&n=" + my;
}
for (const [id] of fields) {
  document.getElementById(id).addEventListener("input", () => {
    clearTimeout(timer);
    timer = setTimeout(refresh, 200);
  });
}
refresh();
</script>
</body>
</html>
"""


class _ExplorerHandler(BaseHTTPRequestHandler):
    def log_message(self, fmt: str, *args) -> None:
        return

    def _send(self, code: int, body: bytes, content_type: str) -> None:
        self.send_response(code)
        self.send_header("Content-Type", content_type)
        self.send_header("Content-Length", str(len(body)))
        self.send_header("Cache-Control", "no-store")
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self) -> None:
        parsed = urlparse(self.path)
        if parsed.path == "/":
            page = (
                _PAGE.replace("__SD__", f"{TEMPORAL_VARIABILITY_THRESHOLD}")
                .replace("__FLAT__", f"{FLATTENING_THRESHOLD}")
                .replace("__SSE__", f"{LOGISTIC_SSE_THRESHOLD}")
                .replace("__AREA__", f"{MIN_CONNECTED_AREA_M2}")
            )
            self._send(200, page.encode(), "text/html; charset=utf-8")
            return
        if parsed.path != "/map.png":
            self._send(404, b"not found", "text/plain; charset=utf-8")
            return
        query = parse_qs(parsed.query)
        try:
            args = tuple(
                float(query[name][0])
                for name in (
                    "temporal_variability_threshold",
                    "flattening_threshold",
                    "logistic_sse_threshold",
                    "min_connected_area_m2",
                )
            )
        except (KeyError, ValueError, IndexError):
            self._send(400, b"bad parameters", "text/plain; charset=utf-8")
            return
        try:
            with _RENDER_LOCK:
                png = render_png(*args)
        except Exception as exc:
            self._send(500, str(exc).encode(), "text/plain; charset=utf-8")
            return
        self._send(200, png, "image/png")


class _ReusableServer(ThreadingHTTPServer):
    allow_reuse_address = True


def serve_explorer() -> str:
    global _SERVER
    try:
        server = _SERVER
    except NameError:
        server = None
    if server is None:
        httpd = _ReusableServer(("127.0.0.1", 8765), _ExplorerHandler)
        threading.Thread(target=httpd.serve_forever, daemon=True).start()
        _SERVER = httpd
        server = httpd
    host, port = server.server_address
    return f"http://{host}:{port}/"


_RENDER_LOCK = threading.Lock()
url = serve_explorer()
print(url)
display(HTML(
    f'<p><a href="{url}" target="_blank">{url}</a></p>'
    f'<iframe src="{url}" title="parameter explorer" width="780" height="860" '
    f'style="border:1px solid #ccc;max-width:100%"></iframe>'
))


http://127.0.0.1:8765/
